In this notebook, we will reimplement LeNet in pytorch.

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

## Download the dataset
In this case, we will use MNIST.

The following cell will download MNIST dataset locally on the first call, and then it will remain available for subsequent calls. **Do not forget to delete the data when you don't need it anymore!**

In [2]:
# Download training data from open datasets.
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.58MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 354kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


Here we specify the size of the batches and create the `DataLoaders`

In [3]:
batch_size = 64

train_dataloader = DataLoader(
    training_data, 
    batch_size=batch_size)

test_dataloader = DataLoader(
    test_data, 
    batch_size=batch_size)

# Sanity check
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break


Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


If you have a GPU available, this is a good moment to use it :)
The following line check for availability and if it is available, it is created.

In [4]:
# The following line will select a GPU (if available) or a cpu for training
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Next, we recreate the `LeNet` architecture from the publication.
For didactic purpose, the convolutional parts are organizes in a `nn.Sequential`, which connects the layers, but the classifier part is instead spelled out manually.

In [5]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_1 = nn.Sequential(
            nn.Conv2d(1,6,kernel_size=5, stride=1, padding=2),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )
        self.conv_2 = nn.Sequential(
            nn.Conv2d(6,16,kernel_size=5,stride=1),
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2)
        )
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(in_features=16*5*5, out_features=120)
        self.tan1 = nn.Tanh()
        self.fc2 = nn.Linear(120,84)
        self.tan2 = nn.Tanh()
        self.fc3 = nn.Linear(84,10)

    def forward(self, x):
        out = self.conv_1(x)
        out = self.conv_2(out)
        out = self.flat(out)
        out = self.fc1(out)
        out = self.tan1(out)
        out = self.fc2(out)
        out = self.tan2(out)
        out = self.fc3(out)
        return out

model = LeNet().to(device)
print(model)

LeNet(
  (conv_1): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (conv_2): Sequential(
    (0): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (tan1): Tanh()
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (tan2): Tanh()
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


We also need to define a loss function (in this case `CrossEntropyLoss`) and an optimizer (in this case `Adam`).
The optimizer should be specified (i) what to optimize: i.e., the parameters and (ii) the learning rate.

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

We then define a training function:

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    # The following line sets the model in `train mode`
    model.train()

    # We then iterate over the dataloader and...
    for batch, (X, y) in enumerate(dataloader):

        # Move the data to the device we want to use
        X, y = X.to(device), y.to(device)

        # Evaluate the model
        pred = model(X)

        # Compute the loss value
        loss = loss_fn(pred, y)

        # Backpropagation - calculate the gradients
        loss.backward()

        # Let the optimizer take a step to improve the parameters
        optimizer.step()

        # Reset the gradient
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

... and a test function

In [8]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    # similarly, here we set the model to `eval` mode, since we do NOT want to calculate the gradient or update the model
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

Next, for a given amount of epochs (passes over the entire dataset), we call the training and testing functions, until we are done. Note how the performance improves with training

In [9]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.315192  [   64/60000]
loss: 0.384335  [ 6464/60000]
loss: 0.298488  [12864/60000]
loss: 0.286244  [19264/60000]
loss: 0.191096  [25664/60000]
loss: 0.202899  [32064/60000]
loss: 0.163391  [38464/60000]
loss: 0.216726  [44864/60000]
loss: 0.240049  [51264/60000]
loss: 0.227216  [57664/60000]
Test Error: 
 Accuracy: 95.8%, Avg loss: 0.133653 

Epoch 2
-------------------------------
loss: 0.081232  [   64/60000]
loss: 0.156548  [ 6464/60000]
loss: 0.123963  [12864/60000]
loss: 0.115678  [19264/60000]
loss: 0.043990  [25664/60000]
loss: 0.070094  [32064/60000]
loss: 0.053571  [38464/60000]
loss: 0.159292  [44864/60000]
loss: 0.186665  [51264/60000]
loss: 0.175744  [57664/60000]
Test Error: 
 Accuracy: 97.5%, Avg loss: 0.082189 

Epoch 3
-------------------------------
loss: 0.022984  [   64/60000]
loss: 0.091502  [ 6464/60000]
loss: 0.096925  [12864/60000]
loss: 0.093365  [19264/60000]
loss: 0.049548  [25664/60000]
loss: 0.053010  [32064/600